# 03 - Exploratory Data Analysis

**Influenza Season Forecasting** - Notebook 3 of 5

**Purpose:** Describe the cleaned data and the targets built in `02_cleaning.ipynb`. This notebook
is **exploratory only**: it defines no new targets, fits no models, and writes/changes no data
files. It saves figures to `figures/`.

**Where the data comes from.** `02_cleaning.ipynb` persists nothing to disk, so this notebook
**reconstructs** the cleaned `season_table` and the per-week smoothed series by re-running 02's
committed, reviewed logic on the raw files (deterministic, identical code). A block of assertions
confirms the reconstruction matches 02 (22 complete seasons, the 5 `holiday_shift` seasons, etc.)
before any plotting. Targets shown here are the **smoothed** targets from 02; raw values appear
only in the holiday-correction QC (view 2).

All descriptions below are **descriptive, not interpretive**. With ~22 seasons, nothing here is a
finding; apparent relationships are hypotheses for later notebooks.

## Setup and reconstruction of the cleaned data (from 02's committed logic)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

DATA_DIR = next((Path(p) for p in ["data/raw", "../data/raw"] if Path(p).exists()), Path("data/raw"))
FIG_DIR = DATA_DIR.parent.parent / "figures"; FIG_DIR.mkdir(exist_ok=True)

PANDEMIC_SEASONS = ["2009-10", "2020-21"]
PANDEMIC_ADJACENT = {"2008-09": "pandemic-adjacent (2009 H1N1 emergence)"}
SMOOTH_WIN = 3
HOLIDAY_WEEKS = {51, 52, 1}
STRAIN_COLORS = {"A(H1N1)": "#1f77b4", "A(H3N2)": "#d62728", "B": "#2ca02c"}
print("figures ->", FIG_DIR.resolve())

In [ ]:
def season_of(year, week):
    start_year = year if week >= 40 else year - 1
    return f"{start_year}-{str(start_year + 1)[2:]}", start_year

# ---- ILINet -> per-week series with season-week index and 3-wk centered smoother ----
ili = pd.read_csv(DATA_DIR / "ILINet.csv", skiprows=1, na_values=["X"])
_i = ili.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
ili["season"] = [x[0] for x in _i]
ili["season_start_year"] = [x[1] for x in _i]
ili["season_order"] = ili["WEEK"].apply(lambda w: w if w >= 40 else w + 100)
ili = ili.sort_values(["season_start_year", "season_order"]).reset_index(drop=True)
ili["season_week"] = ili.groupby("season").cumcount() + 1            # 1 = MMWR wk40
ili["ili_smooth"] = (ili.groupby("season")["% WEIGHTED ILI"]
                        .transform(lambda s: s.rolling(SMOOTH_WIN, center=True).mean()))

# complete seasons: start at wk40, end at wk39
def _complete(g):
    sy = int(g["season_start_year"].iloc[0])
    sp = sorted(g.loc[g["YEAR"] == sy, "WEEK"]); ep = sorted(g.loc[g["YEAR"] == sy + 1, "WEEK"])
    return bool(sp and sp[0] == 40 and ep and ep[0] == 1 and ep[-1] == 39)
complete = [s for s, g in ili.groupby("season") if _complete(g)]
weekly = ili[ili["season"].isin(complete)].copy()

# ---- targets + flags (replicates 02) ----
def peak_on_smoothed(g, win):
    g = g.sort_values("season_order")
    sm = g["% WEIGHTED ILI"].rolling(win, center=True).mean()
    i = sm.idxmax()
    return int(g.loc[i, "WEEK"]), round(float(sm.loc[i]), 3)

rows = []
for s, g in weekly.groupby("season"):
    g = g.sort_values("season_order")
    raw_max = g["% WEIGHTED ILI"].max()
    raw_week = int(g[g["% WEIGHTED ILI"] == raw_max].iloc[0]["WEEK"])
    pk3w, pk3v = peak_on_smoothed(g, SMOOTH_WIN)
    pk5w, _ = peak_on_smoothed(g, 5)
    rows.append({"season": s, "peak_week": pk3w, "peak_ili_pct": pk3v,
                 "peak_week_raw": raw_week, "peak_ili_pct_raw": round(float(raw_max), 3),
                 "peak_week_sm5": pk5w})
season_table = pd.DataFrame(rows).sort_values("season").reset_index(drop=True)
st = season_table
st["holiday_shift"] = st["peak_week_raw"].isin(HOLIDAY_WEEKS) & (st["peak_week"] != st["peak_week_raw"])
st["peak_week_smoothing_sensitive"] = st["peak_week"] != st["peak_week_raw"]
st["fragile_peak_week"] = st["peak_week"] != st["peak_week_sm5"]
st["is_pandemic"] = st["season"].isin(PANDEMIC_SEASONS)
st["special_case"] = st["season"].map(lambda s: "pandemic" if s in PANDEMIC_SEASONS else PANDEMIC_ADJACENT.get(s, ""))

# ---- NREVSS dominant strain stitch (replicates 02) ----
def load_nrevss(fname):
    d = pd.read_csv(DATA_DIR / fname, skiprows=1, na_values=["X", "XX"])
    info = d.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
    d["season"] = [x[0] for x in info]; d["season_start_year"] = [x[1] for x in info]
    return d
def season_dominant(d, source):
    col = lambda n: d[n] if n in d.columns else 0
    bk = pd.DataFrame({"season": d["season"], "season_start_year": d["season_start_year"],
                       "A(H1N1)": col("A (H1)") + col("A (2009 H1N1)"),
                       "A(H3N2)": col("A (H3)"), "B": col("B") + col("BVic") + col("BYam")})
    agg = bk.groupby(["season", "season_start_year"])[["A(H1N1)", "A(H3N2)", "B"]].sum().reset_index()
    agg["dominant_strain"] = agg[["A(H1N1)", "A(H3N2)", "B"]].idxmax(axis=1)
    agg["source"] = source
    return agg
comb = season_dominant(load_nrevss("ICL_NREVSS_Combined_prior_to_2015_16.csv"), "Combined")
phl = season_dominant(load_nrevss("ICL_NREVSS_Public_Health_Labs.csv"), "PublicHealthLabs")
strain = pd.concat([comb[comb["season_start_year"] <= 2014], phl[phl["season_start_year"] >= 2015]]).sort_values("season_start_year").reset_index(drop=True)
st = st.merge(strain[["season", "dominant_strain"]], on="season", how="left")
season_table = st

# ---- prove the reconstruction matches 02 ----
assert len(season_table) == 22
assert sorted(season_table.loc[season_table["holiday_shift"], "season"]) == ["2003-04", "2005-06", "2012-13", "2013-14", "2019-20"]
assert season_table["dominant_strain"].notna().all()
SEASONS = season_table["season"].tolist()
print("reconstruction OK: 22 seasons, holiday_shift=5, strain assigned for all")
print(season_table[["season","peak_week","peak_ili_pct","holiday_shift","fragile_peak_week","is_pandemic","dominant_strain"]].to_string(index=False))

## View 1 - Season trajectory overlay

Every complete season's **smoothed** `% WEIGHTED ILI` on a common x-axis of season-week
(1 = MMWR wk40 ... ~52 = wk39). Normal seasons are thin grey; the two pandemic seasons
(2009-10, 2020-21) and pandemic-adjacent 2008-09 are drawn bold/colored. A small-multiples grid
follows so individual seasons are legible.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
special = {"2009-10": ("#d62728", "-", "2009-10 pandemic"),
           "2020-21": ("#9467bd", "-", "2020-21 pandemic"),
           "2008-09": ("#ff7f0e", "--", "2008-09 pandemic-adjacent")}
for s, g in weekly.groupby("season"):
    g = g.sort_values("season_week")
    if s in special:
        c, ls, lab = special[s]
        ax.plot(g["season_week"], g["ili_smooth"], color=c, ls=ls, lw=2.6, label=lab, zorder=5)
    else:
        ax.plot(g["season_week"], g["ili_smooth"], color="0.6", lw=0.9, alpha=0.6, zorder=1)
ax.plot([], [], color="0.6", lw=0.9, label="normal seasons (19)")
ax.set_xlabel("season week (1 = MMWR wk40)"); ax.set_ylabel("% WEIGHTED ILI (3-wk smoothed)")
ax.set_title("Season trajectory overlay - all 22 complete seasons (smoothed ILI)")
ax.legend(loc="upper right", fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "01a_trajectory_overlay.png", dpi=120); plt.close(fig)
print("saved 01a_trajectory_overlay.png")

In [ ]:
n = len(SEASONS); ncol = 5; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 3 * nrow), sharex=True, sharey=True)
axes = axes.ravel()
for ax, s in zip(axes, SEASONS):
    g = weekly[weekly["season"] == s].sort_values("season_week")
    color = "#d62728" if s in PANDEMIC_SEASONS else ("#ff7f0e" if s in PANDEMIC_ADJACENT else "#1f77b4")
    ax.plot(g["season_week"], g["% WEIGHTED ILI"], color="0.7", lw=0.8, label="raw")
    ax.plot(g["season_week"], g["ili_smooth"], color=color, lw=1.8, label="smoothed")
    row = season_table[season_table["season"] == s].iloc[0]
    tag = " [P]" if row["is_pandemic"] else (" [P-adj]" if row["special_case"] else "")
    ax.set_title(f"{s}{tag}  pk wk{row['peak_week']}={row['peak_ili_pct']:.1f}", fontsize=9)
    ax.grid(alpha=0.3)
for ax in axes[n:]:
    ax.axis("off")
fig.suptitle("Per-season trajectories (grey=raw, colored=smoothed; [P]=pandemic, [P-adj]=pandemic-adjacent)", y=1.002)
fig.supxlabel("season week (1 = MMWR wk40)"); fig.supylabel("% WEIGHTED ILI")
fig.tight_layout(); fig.savefig(FIG_DIR / "01b_trajectory_small_multiples.png", dpi=120, bbox_inches="tight"); plt.close(fig)
print("saved 01b_trajectory_small_multiples.png")

## View 2 - Holiday-correction QC

For the 5 `holiday_shift` seasons, raw vs smoothed `% WEIGHTED ILI`. The raw wk52 point (red ring)
is the single-week year-end spike; the smoothed peak (green triangle) is where the de-spiked target
lands. The smoothed curve should sit visibly below the raw wk52 point and peak elsewhere.

In [ ]:
hs = season_table.loc[season_table["holiday_shift"], "season"].tolist()
fig, axes = plt.subplots(len(hs), 1, figsize=(12, 3.0 * len(hs)))
for ax, s in zip(axes, hs):
    g = weekly[weekly["season"] == s].sort_values("season_week")
    ax.plot(g["season_week"], g["% WEIGHTED ILI"], color="0.55", marker="o", ms=3, lw=1.2, label="raw")
    ax.plot(g["season_week"], g["ili_smooth"], color="#1f77b4", lw=2.2, label="3-wk smoothed")
    row = season_table[season_table["season"] == s].iloc[0]
    w52 = g[g["WEEK"] == 52]
    if len(w52):
        ax.scatter(w52["season_week"], w52["% WEIGHTED ILI"], s=170, facecolors="none",
                   edgecolors="red", linewidths=2, zorder=6, label="raw wk52 (holiday spike)")
    pk = g[g["WEEK"] == row["peak_week"]]
    ax.scatter(pk["season_week"], pk["ili_smooth"], s=130, color="green", marker="^", zorder=6,
               label=f"smoothed peak (wk{row['peak_week']})")
    ax.set_title(f"{s}: raw peak wk{row['peak_week_raw']}={row['peak_ili_pct_raw']:.2f}  ->  smoothed peak wk{row['peak_week']}={row['peak_ili_pct']:.2f}", fontsize=10)
    ax.set_ylabel("% WEIGHTED ILI"); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="upper right")
axes[-1].set_xlabel("season week (1 = MMWR wk40)")
fig.suptitle("Holiday-correction QC - wk52 spike removal on the 5 holiday_shift seasons", y=1.001)
fig.tight_layout(); fig.savefig(FIG_DIR / "02_holiday_correction_qc.png", dpi=120, bbox_inches="tight"); plt.close(fig)
print("saved 02_holiday_correction_qc.png")
# numeric confirmation
qc = season_table[season_table["holiday_shift"]][["season","peak_week_raw","peak_ili_pct_raw","peak_week","peak_ili_pct"]].copy()
qc["height_removed"] = (qc["peak_ili_pct_raw"] - qc["peak_ili_pct"]).round(3)
qc["week_moved"] = qc["peak_week_raw"].astype(str) + " -> " + qc["peak_week"].astype(str)
print(qc.to_string(index=False))

## View 3 - peak_week distribution (non-pandemic seasons)

Strip plot of the smoothed `peak_week` across the non-pandemic seasons (x = MMWR week in season
order). Seasons carrying `fragile_peak_week` (3-wk vs 5-wk smoother disagree on timing) are marked
with an open red ring: their week label is uncertain to about +/-1 week.

In [ ]:
np_tbl = season_table[~season_table["is_pandemic"]].copy()
np_tbl["order"] = np_tbl["peak_week"].apply(lambda w: w if w >= 40 else w + 100)
fig, ax = plt.subplots(figsize=(13, 3.4))
jit = np.random.default_rng(0).uniform(-0.12, 0.12, len(np_tbl))
ax.scatter(np_tbl["order"], jit, s=70, color="#1f77b4", zorder=3, label="non-pandemic season")
frag = np_tbl[np_tbl["fragile_peak_week"]]
ax.scatter(frag["order"], jit[np_tbl["fragile_peak_week"].values], s=180, facecolors="none",
           edgecolors="red", linewidths=1.8, zorder=4, label="fragile_peak_week")
for _, r in np_tbl.iterrows():
    ax.annotate(r["season"][2:], (r["order"], 0), fontsize=6, rotation=90, ha="center", va="bottom", color="0.4")
ticks = list(range(40, 54)) + [100 + w for w in range(1, 16)]
ax.set_xticks(ticks); ax.set_xticklabels([str(t if t < 54 else t - 100) for t in ticks], fontsize=7)
ax.set_yticks([]); ax.set_xlabel("peak_week (MMWR week, season order wk40..wk39)")
ax.set_title(f"Smoothed peak_week across {len(np_tbl)} non-pandemic seasons"); ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "03_peak_week_distribution.png", dpi=120); plt.close(fig)
print("saved 03_peak_week_distribution.png")
print("fragile among non-pandemic:", frag["season"].tolist())

## View 4 - peak_ili_pct distribution (non-pandemic seasons)

Histogram of smoothed `peak_ili_pct` across the non-pandemic seasons, with the individual season
values as a rug beneath.

In [ ]:
vals = np_tbl["peak_ili_pct"].values
fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(vals, bins=np.arange(2.0, 8.5, 0.5), color="#1f77b4", alpha=0.8, edgecolor="white")
for v in vals:
    ax.plot([v, v], [-0.18, -0.02], color="0.3", lw=1)
ax.axvline(np.median(vals), color="red", ls="--", lw=1.5, label=f"median {np.median(vals):.2f}")
ax.set_xlabel("peak_ili_pct (smoothed, % WEIGHTED ILI)"); ax.set_ylabel("number of seasons")
ax.set_title(f"Smoothed peak_ili_pct across {len(vals)} non-pandemic seasons"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "04_peak_ili_distribution.png", dpi=120); plt.close(fig)
print("saved 04_peak_ili_distribution.png")
print("min/median/max peak_ili_pct (non-pandemic): {:.2f} / {:.2f} / {:.2f}".format(vals.min(), np.median(vals), vals.max()))

## View 5 - Dominant-strain timeline

One row, 22 seasons in chronological order, each cell colored by the stitched-NREVSS dominant
strain. Pandemic / pandemic-adjacent seasons are marked with a hatch and an asterisk.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 2.4))
for i, s in enumerate(SEASONS):
    row = season_table[season_table["season"] == s].iloc[0]
    ds = row["dominant_strain"]
    hatch = "//" if (row["is_pandemic"] or row["special_case"]) else None
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=STRAIN_COLORS[ds], edgecolor="white", hatch=hatch))
    star = "*" if (row["is_pandemic"] or row["special_case"]) else ""
    ax.text(i + 0.5, 0.5, ds.replace("A(", "").replace(")", "") + star, ha="center", va="center",
            fontsize=7, color="white", rotation=90, weight="bold")
ax.set_xlim(0, len(SEASONS)); ax.set_ylim(0, 1)
ax.set_xticks(np.arange(len(SEASONS)) + 0.5); ax.set_xticklabels(SEASONS, rotation=90, fontsize=8)
ax.set_yticks([]); ax.set_title("Dominant strain by season (stitched NREVSS); * = pandemic / pandemic-adjacent")
ax.legend(handles=[Patch(facecolor=c, label=k) for k, c in STRAIN_COLORS.items()], ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.35))
fig.tight_layout(); fig.savefig(FIG_DIR / "05_strain_timeline.png", dpi=120, bbox_inches="tight"); plt.close(fig)
print("saved 05_strain_timeline.png")
print(season_table.set_index("season")["dominant_strain"].to_dict())

## View 6 - peak_ili_pct by dominant strain

**DESCRIPTIVE ONLY.** n is ~22 seasons (here 20 non-pandemic), and the groups are small and
unbalanced. Any apparent strain/severity pattern is a **hypothesis for a later notebook, not a
finding**. Pandemic seasons are excluded; counts per strain are printed so the reader sees how thin
each group is.

In [ ]:
sub = season_table[~season_table["is_pandemic"]]
groups = ["A(H1N1)", "A(H3N2)", "B"]
data = [sub.loc[sub["dominant_strain"] == gv, "peak_ili_pct"].values for gv in groups]
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(data, labels=[f"{g}\n(n={len(d)})" for g, d in zip(groups, data)], showfliers=False, widths=0.5)
rng = np.random.default_rng(1)
for i, (gv, d) in enumerate(zip(groups, data), start=1):
    ax.scatter(np.full(len(d), i) + rng.uniform(-0.12, 0.12, len(d)), d, color=STRAIN_COLORS[gv], s=55, zorder=3, alpha=0.85)
ax.set_ylabel("peak_ili_pct (smoothed)"); ax.set_xlabel("dominant strain")
ax.set_title("peak_ili_pct by dominant strain - DESCRIPTIVE ONLY (small n, not a finding)"); ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "06_peakili_by_strain.png", dpi=120); plt.close(fig)
print("saved 06_peakili_by_strain.png")
for g, d in zip(groups, data):
    print(f"  {g}: n={len(d)}  peak_ili median={np.median(d):.2f}  range {d.min():.2f}-{d.max():.2f}")

## View 7 - Missingness map across sources

Per-season availability of the three time-varying sources. Cell value = fraction of in-season
weeks with a missing value (ILINet `% WEIGHTED ILI`; FluSurv-NET `WEEKLY RATE`). Absent =
source does not cover that season at all (the enrichment sources start 2009-10). FluVaxView is a
single season-end value, shown as present/absent.

In [ ]:
# ILINet missingness per season
ili_miss = (weekly.groupby("season")["% WEIGHTED ILI"]
            .apply(lambda s: s.isna().mean()).reindex(SEASONS))

# FluSurv-NET (season label already in YEAR)
fs = pd.read_csv(DATA_DIR / "FluSurveillance_Custom_Download_Data.csv", skiprows=2, na_values=["null"])
fs.columns = [c.strip() for c in fs.columns]
fs = fs[(fs["CATCHMENT"] == "Entire Network") & (fs["AGE CATEGORY"] == "Overall") &
        (fs["SEX CATEGORY"] == "Overall") & (fs["RACE CATEGORY"] == "Overall") & (fs["VIRUS TYPE CATEGORY"] == "Overall")]
fs_frac = fs.groupby("YEAR")["WEEKLY RATE"].apply(lambda s: s.isna().mean())
fs_miss = pd.Series({s: (fs_frac[s] if s in fs_frac.index else np.nan) for s in SEASONS})

# FluVaxView present/absent
ALLAGES = [">=6 Months", "Greater than 6 Months flu"]
keep = []
for ch in pd.read_csv(DATA_DIR / "FluVaxView.csv", chunksize=200_000, dtype=str):
    m = ch[(ch["Geography"] == "United States") & (ch["Dimension Type"] == "Age") & (ch["Dimension"].isin(ALLAGES))]
    if len(m): keep.append(m)
if not keep:
    raise ValueError("No FluVaxView rows matched the national all-ages filter; "
                     "the '>=6 Months' Dimension label may have changed again (extend ALLAGES).")
vx = pd.concat(keep); vx["e"] = pd.to_numeric(vx["Estimate (%)"], errors="coerce")
vx_seasons = set(vx.dropna(subset=["e"])["Season/Survey Year"].unique())
vax_miss = pd.Series({s: (0.0 if s in vx_seasons else np.nan) for s in SEASONS})

M = pd.DataFrame({"ILINet": ili_miss, "FluSurv-NET": fs_miss, "FluVaxView": vax_miss}).reindex(SEASONS)
fig, ax = plt.subplots(figsize=(7.5, 9))
cmap = plt.cm.Reds.copy(); cmap.set_bad("0.85")   # absent -> grey
im = ax.imshow(M.values, aspect="auto", cmap=cmap, vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(M.columns)
ax.set_yticks(range(len(SEASONS))); ax.set_yticklabels(SEASONS, fontsize=8)
for i in range(len(SEASONS)):
    for j in range(3):
        v = M.values[i, j]
        txt = "absent" if np.isnan(v) else ("ok" if v == 0 else f"{v:.0%}")
        ax.text(j, i, txt, ha="center", va="center", fontsize=7, color="0.3" if (np.isnan(v) or v < 0.5) else "white")
ax.set_title("Missingness by season and source\n(grey=source absent; red=fraction of weeks missing)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="fraction of in-season weeks missing")
fig.tight_layout(); fig.savefig(FIG_DIR / "07_missingness_map.png", dpi=120); plt.close(fig)
print("saved 07_missingness_map.png")
print("ILINet weeks missing target:", int((weekly["% WEIGHTED ILI"].isna()).sum()))
print("FluSurv-NET seasons with any missing WEEKLY RATE:", fs_frac[fs_frac > 0].index.tolist())
print("FluVaxView seasons present:", len(vx_seasons), "| FluSurv-NET seasons present:", int(fs_miss.notna().sum()))

## Next step

`04_baselines.ipynb`: naive baselines (historical-median peak week; prior-season / historical-mean
peak ILI) that every later model must beat. No modeling happens before those baselines exist.